## Variable de Control
Diego Dueñas Martín   
exp 751426  
21/10/2026

### Opción asiática con variable de control

Se simulan 100 trayectorias mensuales del precio del activo y se calcula el payoff de una call asiática aritmética. Como variable de control, se usa una call asiática geométrica, cuyo precio exacto se conoce. Finalmente, se comparan las estimaciones con y sin variable de control y sus errores estándar. Se muestran también las primeras 10 trayectorias.

In [1]:
import math
import numpy as np
import pandas as pd

S0 = 100
K = 100
sigma = 0.20
r = 0.05
T = 1
n_rutas = 100
n_observaciones = 12

rng = np.random.default_rng(42)
dt = T / n_observaciones
Z = rng.standard_normal((n_rutas, n_observaciones))
log_retornos = (r - sigma**2 / 2) * dt + sigma * np.sqrt(dt) * Z
precios = S0 * np.exp(np.cumsum(log_retornos, axis=1))

promedio_aritmetico = precios.mean(axis=1)
promedio_geometrico = np.exp(np.log(precios).mean(axis=1))
descuento = np.exp(-r * T)
payoff_aritmetico = descuento * np.maximum(promedio_aritmetico - K, 0)
payoff_geometrico = descuento * np.maximum(promedio_geometrico - K, 0)

# El promedio geometrico tiene lognormalidad; se usa su precio exacto como E[Y].
tiempos = dt * np.arange(1, n_observaciones + 1)
media_log_geometrica = np.log(S0) + (r - sigma**2 / 2) * tiempos.mean()
var_log_geometrica = sigma**2 * np.minimum.outer(tiempos, tiempos).sum() / n_observaciones**2
sd_log_geometrica = np.sqrt(var_log_geometrica)
d1 = (media_log_geometrica - np.log(K) + var_log_geometrica) / sd_log_geometrica
d2 = d1 - sd_log_geometrica
N = lambda x: 0.5 * (1 + math.erf(x / np.sqrt(2)))
precio_geometrico_exacto = np.exp(-r * T) * (
    np.exp(media_log_geometrica + var_log_geometrica / 2) * N(d1) - K * N(d2)
)

# Estimador de control: X_bar - beta * (Y_bar - E[Y]).
beta = np.cov(payoff_aritmetico, payoff_geometrico, ddof=1)[0, 1] / np.var(payoff_geometrico, ddof=1)
estimacion_sin_control = payoff_aritmetico.mean()
estimacion_con_control = estimacion_sin_control - beta * (payoff_geometrico.mean() - precio_geometrico_exacto)
error_estandar_sin_control = payoff_aritmetico.std(ddof=1) / np.sqrt(n_rutas)
error_estandar_con_control = (payoff_aritmetico - beta * payoff_geometrico).std(ddof=1) / np.sqrt(n_rutas)

print(f"Precio geometrico exacto (control): {precio_geometrico_exacto:.4f}")
print(f"Beta estimada: {beta:.4f}")
print(f"Call asiatica aritmetica sin control: {estimacion_sin_control:.4f} (EE: {error_estandar_sin_control:.4f})")
print(f"Call asiatica aritmetica con control: {estimacion_con_control:.4f} (EE: {error_estandar_con_control:.4f})")
correlacion = np.corrcoef(payoff_aritmetico, payoff_geometrico)[0, 1]
factor_varianza = (error_estandar_sin_control / error_estandar_con_control) ** 2
print(f"Correlacion entre payoffs: {correlacion:.4f}")
print(f"Reduccion de varianza: {factor_varianza:.0f} veces")

Precio geometrico exacto (control): 5.9402
Beta estimada: 1.0362
Call asiatica aritmetica sin control: 5.3316 (EE: 0.7744)
Call asiatica aritmetica con control: 6.1847 (EE: 0.0202)
Correlacion entre payoffs: 0.9997
Reduccion de varianza: 1476 veces
